# 🫀 MASH Pipeline — Visit-Level CSV Builder
> **One row per visit · All quotes saved · Ollama (qwen3:8b) · JupyterLab**
>
> **v3 changes:** N-patient split — Cell 6b auto-detects distinct patients by DOB and writes one CSV + JSON per patient

| Cell | Purpose |
|------|---------|
| 1 | Config (edit paths here) |
| 2 | OCR — extract text from PDF |
| 3 | Segment — split into individual visits |
| 4 | Extract — Ollama call per visit |
| 5 | Review — interactive ipywidgets table |
| 6 | Save — write combined CSV + JSON (all visits) |
| 6b | **Split — write one CSV + JSON per patient (N patients)** |
| 7 | Plot — timeline of key markers (optional) |


## ⚙️ Cell 1 — Config  *(only cell you need to edit)*

In [1]:
import sys, os, re, json, math, csv, copy
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output

# ── Paths ─────────────────────────────────────────────────────
MODULES_DIR  = Path("medical_ocr_system")   # folder containing ocr.py etc.
REPORT_FILE  = "EHR2.pdf"               # ← change per run
OUTPUT_DIR   = Path("output")

# ── Ollama ────────────────────────────────────────────────────
OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL        = "qwen3:8b"
TEMPERATURE  = 0.05
TIMEOUT      = 900          # seconds per Ollama call
NUM_PREDICT  = 5000         # tokens for JSON output (22 fields × sub-object)
THINK_MODE   = False        # qwen3: False disables <think> chain-of-thought
                            # Set True only if you want reasoning (burns tokens)

# ── Sliding window (fallback for very long visit chunks) ──────
WINDOW_LINES    = 60        # lines per window
WINDOW_OVERLAP  = 15        # overlap between windows
WINDOW_TRIGGER  = 80        # use sliding window if chunk > this many lines

# ── Tesseract / Poppler (Windows paths — adjust as needed) ────
os.environ["POPPLER_PATH"]  = r"C:\Users\evang\Downloads\Release-26.02.0-0\poppler-26.02.0\Library\bin"
os.environ["TESSERACT_CMD"] = r"C:\Users\evang\Downloads\Ball\tesseract.exe"

# ── Load OCR modules ──────────────────────────────────────────
if MODULES_DIR.exists():
    sys.path.insert(0, str(MODULES_DIR))
    from modules import ocr as ocr_mod
    print(f"✅ Modules loaded from {MODULES_DIR.resolve()}")
else:
    print(f"⚠  Modules dir not found at {MODULES_DIR.resolve()}")
    print("   OCR unavailable — you can still run from cached text.")

print(f"Report : {REPORT_FILE}")
print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}")

✅ Modules loaded from C:\Users\evang\Desktop\hackathon\medical_ocr_system
Report : EHR2.pdf
Model  : qwen3:8b
Output : output


## 📄 Cell 2 — OCR

In [2]:
report_path = Path(REPORT_FILE)
if not report_path.exists():
    raise FileNotFoundError(f"Report not found: {report_path.resolve()}")

print(f"Processing: {report_path.name}  ({report_path.stat().st_size/1024:.1f} KB) ...")
ocr_result = ocr_mod.extract(report_path)
FULL_TEXT  = ocr_result.full_text

print(f"\nOCR engine : {ocr_result.source}")
print(f"Pages      : {len(ocr_result.page_map)}")
print(f"Characters : {len(FULL_TEXT):,}")
for w in ocr_result.warnings:
    print(f"⚠  {w}")

raw_path = OUTPUT_DIR / f"{report_path.stem}_ocr_raw.txt"
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw_path.write_text(FULL_TEXT, encoding="utf-8")
print(f"\nRaw OCR text saved → {raw_path}")
print("\n── First 500 chars ──")
print(FULL_TEXT[:500])

Processing: EHR2.pdf  (1170.1 KB) ...

OCR engine : tesseract
Pages      : 9
Characters : 14,233

Raw OCR text saved → output\EHR2_ocr_raw.txt

── First 500 chars ──
>>>>>>>>> >>> >>> >> >>> >> >> >>> >>> >>> >>> Note | <<<<<<<<<ccccéceeeécccé§y§eeéccéé§y§¥§¥<<<§<$<
Note index: 1

Note Date: 2024-04-10

Date of Birth: 1981-02-14

Note:

Date of service: April 10, 2024

Reason for visit: Hepatology follow-up for metabolic dysfunction-associated steatohepatitis (MASH) and type
2 diabetes mellitus. Fibrosis staging with transient elastography today.

Diagnosis:

Metabolic Dysfunction-Associated Steatohepatitis (MASH) with Moderate Fibrosis (F2 by FibroScan)
His


## 🔪 Cell 3 — Visit Segmenter
Splits the document on `Note N` headers. Confirm the table looks right before running extraction.

In [3]:
# ── Patterns ─────────────────────────────────────────────────
# Matches:
# ================================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>> Note 2 <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ================================================================================
# Note index: 2
# Note Date: 2024-11-15
# Date of Birth: 1969-03-20

NOTE_INDEX_RE = re.compile(r"^Note\s+index\s*:\s*(\d+)", re.MULTILINE | re.IGNORECASE)
NOTE_DATE_RE  = re.compile(r"^Note\s+Date\s*:\s*(\S+)",   re.MULTILINE | re.IGNORECASE)
NOTE_DOB_RE   = re.compile(r"^Date\s+of\s+Birth\s*:\s*(\S+)", re.MULTILINE | re.IGNORECASE)
BORDER_RE     = re.compile(r"^[=>{<\-]{20,}\s*$")


def _walk_back_to_border(text, pos):
    """Walk back from pos to capture the === header lines above a Note."""
    cur = text.rfind("\n", 0, pos)
    for _ in range(8):
        prev = text.rfind("\n", 0, cur)
        if prev < 0:
            return 0
        line = text[prev + 1:cur].strip()
        if BORDER_RE.match(line) or (line and ("Note" in line or "<<" in line or ">>" in line)):
            cur = prev
        else:
            break
    return max(0, cur + 1)


def segment_visits(text):
    """
    Returns list of visit dicts:
      { note_index, note_date, dob, char_start, char_end, text, line_count }
    Split points are the "Note index:" lines.
    """
    anchors = [(m.start(), int(m.group(1))) for m in NOTE_INDEX_RE.finditer(text)]

    if not anchors:
        print("⚠  No 'Note index:' markers found — treating whole text as one visit.")
        return [_build_visit(1, text, 0, len(text))]

    visits = []
    for i, (pos, idx) in enumerate(anchors):
        block_start = _walk_back_to_border(text, pos)
        if i + 1 < len(anchors):
            block_end = _walk_back_to_border(text, anchors[i + 1][0])
        else:
            block_end = len(text)
        chunk = text[block_start:block_end]
        visits.append(_build_visit(idx, chunk, block_start, block_end))
    return visits


def _build_visit(note_index, chunk, char_start, char_end):
    date_m = NOTE_DATE_RE.search(chunk)
    dob_m  = NOTE_DOB_RE.search(chunk)
    return {
        "note_index" : note_index,
        "note_date"  : date_m.group(1) if date_m else None,
        "dob"        : dob_m.group(1)  if dob_m  else None,
        "char_start" : char_start,
        "char_end"   : char_end,
        "text"       : chunk,
        "line_count" : len(chunk.splitlines()),
    }


# ── Run ────────────────────────────────────────────────────────
VISITS = segment_visits(FULL_TEXT)
print(f"Found {len(VISITS)} visit(s)\n")

# ── Change 3: DOB consistency check ──────────────────────────
dobs = {v["dob"] for v in VISITS if v["dob"]}
if len(dobs) > 1:
    print(f"⚠  WARNING: Multiple DOBs detected across notes: {dobs}")
    print("   This PDF may contain notes for more than one patient.")
    print("   Review the segmentation table below carefully.\n")
elif len(dobs) == 1:
    print(f"✅ Single patient DOB confirmed: {next(iter(dobs))}\n")

rows_html = "".join(
    f'<tr>'
    f'<td style="text-align:center">{v["note_index"]}</td>'
    f'<td>{v["note_date"] or "?"}</td>'
    f'<td>{v["dob"] or "?"}</td>'
    f'<td style="text-align:right">{v["line_count"]}</td>'
    f'<td style="color:{"#856404" if v["line_count"] > WINDOW_TRIGGER else "#155724"};">'
    f'{"⚠ sliding window" if v["line_count"] > WINDOW_TRIGGER else "✓ direct"}</td>'
    f'</tr>'
    for v in VISITS
)
display(HTML(
    '<table border="1" cellpadding="6" style="border-collapse:collapse;font-size:.9em">'
    '<thead><tr style="background:#343a40;color:white">'
    '<th>#</th><th>Note Date</th><th>DOB</th><th>Lines</th><th>Strategy</th>'
    '</tr></thead><tbody>' + rows_html + '</tbody></table>'
))

# Optionally preview a specific visit chunk:
# print(VISITS[0]["text"][:800])

Found 2 visit(s)

⚠  WARNING: Multiple DOBs detected across notes: {'1969-03-20', '1981-02-14'}
   This PDF may contain notes for more than one patient.
   Review the segmentation table below carefully.



#,Note Date,DOB,Lines,Strategy
1,2024-04-10,1981-02-14,233,⚠ sliding window
2,2024-11-15,1969-03-20,242,⚠ sliding window


## 🤖 Cell 4 — Ollama Extraction
Calls `qwen3:8b` once per visit (or via sliding window for long visits).
Results stored in `VISIT_ROWS` — one dict per visit, with `_quote` for every field.

In [4]:
import requests

# ── Field schema ─────────────────────────────────────────────
# Every field gets:  field, field_unit, field_ref_range, field_quote, field_confidence
LIVER_FIELDS = [
    "visit_date",
    "age",
    "gender",
    "ast",
    "alt",
    "ggt",
    "alp",
    "bilirubin_total",
    "bilirubin_direct",
    "albumin",
    "platelets",
    "inr",
    "creatinine",
    "glucose",
    "hba1c",
    "fibroscan_kpa",
    "fibroscan_cap",
    "fibrosis_stage",
    "nas_score",
    "fib4_reported",
    "diagnosis",
    "medications",
    # ── NEW ──────────────────────────────
    "icd9_code",
    "short_title",
    "long_title",
    "liver_group",
]

SYSTEM_PROMPT = """\
You are a precise medical data extractor specialising in liver disease / MASH.

MASH context:
- MASH = Metabolic dysfunction-Associated Steatohepatitis (formerly NASH / NAFLD).
- Fibrosis staged F0-F4 (F4 = cirrhosis). FIB-4 = (Age x AST) / (Platelets x sqrt(ALT)).
- FibroScan: liver stiffness kPa; CAP score dB/m.
- Key labs: ALT, AST, GGT, ALP, bilirubin, albumin, platelets, INR, creatinine, HbA1c.
- icd9_code: ICD-9 diagnosis code (e.g. 571.8, 571.5).
- short_title: short ICD-9 title (e.g. "Other chronic nonalcoholic liver disease").
- long_title: full ICD-9 description if present.
- liver_group: disease group/category label (e.g. MASH, NAFLD, ALD, cirrhosis, steatosis).

STRICT RULES:
1. Return ONLY a single valid JSON object. No markdown, no preamble.
2. Keys are exactly the field names requested. Every field must appear.
3. Each field value must be this sub-object:
   {"value":<str|null>,"unit":<str|null>,"reference_range":<str|null>,"exact_quote":<verbatim|null>,"confidence":"high"|"medium"|"low"}
4. exact_quote must be copied VERBATIM from the source text — never invented.
5. If a field is absent: value=null, exact_quote=null, confidence="low".
6. When a field appears multiple times, use the most recent value.

Example (two fields):
{"ast":{"value":"71","unit":"U/L","reference_range":"14-40","exact_quote":"AST 71 14-40 U/L (H)","confidence":"high"},"albumin":{"value":null,"unit":null,"reference_range":null,"exact_quote":null,"confidence":"low"}}
"""


def strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def call_ollama(chunk_text, fields, label=""):
    """Single Ollama call. Returns dict keyed by field name."""
    prompt = (
        "/no_think\n"
        f"Extract these fields: {json.dumps(fields)}\n\n"
        f"Report text:\n{chunk_text}\n\n"
        "Return ONLY the JSON object."
    )
    payload = {
        "model":   MODEL,
        "system":  SYSTEM_PROMPT,
        "prompt":  prompt,
        "stream":  False,
        "think":   THINK_MODE,          # qwen3: disables <think> budget drain
        "options": {"temperature": TEMPERATURE, "num_predict": NUM_PREDICT},
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
        resp.raise_for_status()
        raw = strip_think(resp.json().get("response", ""))
        if label.endswith("/w1") or not "/" in label:  # debug first call per visit
            preview = raw[:300].replace("\n", " ")
            print(f"    🔍 raw[{label}]: {preview}")
        raw = re.sub(r"```(?:json)?", "", raw)
        raw = re.sub(r"```", "", raw).strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            raw = m.group(0)
        return json.loads(raw)
    except Exception as e:
        print(f"    ❌ Ollama error [{label}]: {e}")
        null_sub = {"value":None,"unit":None,"reference_range":None,"exact_quote":None,"confidence":"low"}
        return {f: null_sub.copy() for f in fields}


def extract_sliding_window(chunk_text, fields, label):
    """Overlapping windows — last non-null value per field wins (most recent)."""
    lines  = chunk_text.splitlines()
    step   = WINDOW_LINES - WINDOW_OVERLAP
    CONF_RANK = {"high": 3, "medium": 2, "low": 1}
    merged  = {}  # f -> best sub seen so far
    w_num   = 0
    w_found = {}  # f -> window number where best value came from
    for i in range(0, len(lines), step):
        w_num += 1
        w_text = "\n".join(lines[i:i + WINDOW_LINES])
        result = call_ollama(w_text, fields, label=f"{label}/w{w_num}")
        for f, sub in result.items():
            if not isinstance(sub, dict):
                continue
            val  = sub.get("value")
            if not val or str(val).strip().lower() in ("", "null", "none", "n/a"):
                continue
            new_conf  = CONF_RANK.get(sub.get("confidence", "low"), 1)
            prev_conf = CONF_RANK.get(merged.get(f, {}).get("confidence", "low"), 0)
            if new_conf > prev_conf:   # only upgrade confidence
                merged[f] = sub
                w_found[f] = w_num
        if i + WINDOW_LINES >= len(lines):
            break
    # Summary of what each window contributed
    if w_found:
        summary = ", ".join(f"{f}←w{w}" for f, w in sorted(w_found.items()))
        print(f"    📋 best-source map: {summary}")
    null_sub = {"value":None,"unit":None,"reference_range":None,"exact_quote":None,"confidence":"low"}
    for f in fields:
        merged.setdefault(f, null_sub.copy())
    print(f"    ↳ {w_num} windows")
    return merged


def _safe_float(val):
    if val is None: return None
    try: return float(re.sub(r"[^\d.]", "", str(val)) or "x")
    except ValueError: return None


def extract_visit(visit):
    """Extract one visit → flat row dict with value + _quote columns."""
    label = f"Note {visit['note_index']}"

    if visit["line_count"] > WINDOW_TRIGGER:
        raw = extract_sliding_window(visit["text"], LIVER_FIELDS, label)
    else:
        raw = call_ollama(visit["text"], LIVER_FIELDS, label=label)

    row = {
    "subject_id":  visit["note_index"],   # ← ADD THIS LINE
    "_note_index": visit["note_index"],
    "_note_date":  visit["note_date"],
    "_dob":        visit["dob"],
    "_char_start": visit["char_start"],
    "_char_end":   visit["char_end"],
    }

    for f in LIVER_FIELDS:
        sub = raw.get(f) or {}
        if not isinstance(sub, dict):
            sub = {"value": str(sub), "unit": None, "reference_range": None,
                   "exact_quote": None, "confidence": "medium"}
        row[f]                 = sub.get("value")
        row[f + "_unit"]       = sub.get("unit")
        row[f + "_ref_range"]  = sub.get("reference_range")
        row[f + "_quote"]      = sub.get("exact_quote")
        row[f + "_confidence"] = sub.get("confidence", "low")

    # ── Computed: FIB-4 ───────────────────────────────────────
    ast = _safe_float(row.get("ast"))
    alt = _safe_float(row.get("alt"))
    age = _safe_float(row.get("age"))
    plt = _safe_float(row.get("platelets"))
    if ast and alt and age and plt and plt > 0 and alt > 0:
        fib4 = round((age * ast) / (plt * math.sqrt(alt)), 3)
        row["fib4_computed"]       = fib4
        row["fib4_computed_quote"] = f"COMPUTED: (Age {age} x AST {ast}) / (Plt {plt} x sqrt(ALT {alt})) = {fib4}"
    else:
        row["fib4_computed"]       = None
        row["fib4_computed_quote"] = None

    # ── Computed: AST/ALT ratio ───────────────────────────────
    if ast and alt and alt != 0:
        row["ast_alt_ratio"]       = round(ast / alt, 3)
        row["ast_alt_ratio_quote"] = f"COMPUTED: AST {ast} / ALT {alt}"
    else:
        row["ast_alt_ratio"]       = None
        row["ast_alt_ratio_quote"] = None

    return row


# ── Main extraction loop ───────────────────────────────────────
print(f"Extracting {len(VISITS)} visit(s) with {MODEL} ...\n")
VISIT_ROWS = []

try:
    from tqdm.notebook import tqdm as _tqdm
    _iter = _tqdm(VISITS, desc="Visits")
except ImportError:
    _iter = VISITS

for visit in _iter:
    info = f"Note {visit['note_index']} | {visit['note_date'] or '?'} | {visit['line_count']} lines"
    print(f"  → {info}")
    VISIT_ROWS.append(extract_visit(visit))

print(f"\n✅ Done — {len(VISIT_ROWS)} visit row(s) extracted.")

Extracting 2 visit(s) with qwen3:8b ...



Visits:   0%|          | 0/2 [00:00<?, ?it/s]

  → Note 1 | 2024-04-10 | 233 lines
    🔍 raw[Note 1/w1]: {   "visit_date": {"value":"2024-04-10","unit":null,"reference_range":null,"exact_quote":"Note Date: 2024-04-10","confidence":"high"},   "age": {"value":"43","unit":"years","reference_range":null,"exact_quote":"43-year-old male","confidence":"high"},   "gender": {"value":"male","unit":null,"referenc
    📋 best-source map: age←w1, albumin←w4, alp←w4, alt←w4, ast←w4, bilirubin_total←w4, creatinine←w4, diagnosis←w1, fib4_reported←w5, fibroscan_cap←w5, fibroscan_kpa←w1, fibrosis_stage←w1, gender←w1, glucose←w4, hba1c←w5, liver_group←w1, medications←w1, platelets←w3, visit_date←w1
    ↳ 5 windows
  → Note 2 | 2024-11-15 | 242 lines
    🔍 raw[Note 2/w1]: {   "visit_date": {"value":"2024-11-15","unit":null,"reference_range":null,"exact_quote":"Note Date: 2024-11-15","confidence":"high"},   "age": {"value":"55","unit":"years","reference_range":null,"exact_quote":"55-year-old female","confidence":"high"},   "gender": {"value":"female","u

## 🔍 Cell 5 — Interactive Review
Edit any extracted value before saving. Colour key:
🟢 **green** = high confidence + quote verified  🟡 **amber** = medium confidence or fuzzy quote  🔴 **red** = not found or quote unverifiable

In [5]:
import ipywidgets as widgets

DISPLAY_FIELDS = LIVER_FIELDS + ["fib4_computed", "ast_alt_ratio"]

# Mutable copy for editing
EDITED_ROWS = copy.deepcopy(VISIT_ROWS)


def _sanity(quote, full_text):
    if not quote or str(quote).strip().lower() in ("", "none", "null"):
        return "SKIPPED"
    q = str(quote)
    if full_text.find(q) >= 0 or full_text.lower().find(q.lower()) >= 0:
        return "VERIFIED"
    if full_text.lower().find(q[:50].lower()) >= 0:
        return "FUZZY"
    return "NOT_FOUND"


def _cell_bg(value, confidence, quote):
    if value is None or str(value).strip() in ("", "None", "null"):
        return "#f8d7da"
    s = _sanity(quote, FULL_TEXT)
    if s == "NOT_FOUND":
        return "#f8d7da"
    if confidence == "high" and s == "VERIFIED":
        return "#d4edda"
    return "#fff3cd"


# ── Build widget grid ─────────────────────────────────────────
widget_refs = {}   # (row_idx, field) → widget

def _hdr(text):
    return widgets.HTML(
        f'<div style="background:#343a40;color:white;padding:4px 8px;'
        f'font-weight:bold;font-size:.82em;min-width:95px">{text}</div>')

header_row = widgets.HBox(
    [_hdr("Note #"), _hdr("Date")] + [_hdr(f) for f in DISPLAY_FIELDS]
)

grid_rows = [header_row]
for r_idx, row in enumerate(EDITED_ROWS):
    row_widgets = [
        widgets.HTML(f'<div style="padding:4px 8px;font-size:.82em;min-width:55px">'
                     f'{row.get("_note_index","")}</div>'),
        widgets.HTML(f'<div style="padding:4px 8px;font-size:.82em;min-width:90px">'
                     f'{row.get("_note_date","")}</div>'),
    ]
    for field in DISPLAY_FIELDS:
        val  = row.get(field)
        conf = row.get(field + "_confidence", "low")
        qte  = row.get(field + "_quote")
        bg   = _cell_bg(val, conf, qte)

        w = widgets.Text(
            value       = str(val) if val is not None else "",
            placeholder = "—",
            layout      = widgets.Layout(width="105px"),
            style       = {"description_width": "0px"},
        )

        def _make_obs(ri, fi):
            def _obs(change):
                EDITED_ROWS[ri][fi] = change["new"].strip() or None
            return _obs
        w.observe(_make_obs(r_idx, field), names="value")
        widget_refs[(r_idx, field)] = w

        row_widgets.append(
            widgets.Box([w], layout=widgets.Layout(
                background_color=bg, padding="2px", margin="1px"
            ))
        )
    grid_rows.append(widgets.HBox(row_widgets))

print("Edit any cell. Changes update EDITED_ROWS immediately.")
print("🟢 green = verified  🟡 amber = fuzzy/medium  🔴 red = not found\n")
display(widgets.VBox(grid_rows, layout=widgets.Layout(overflow_x="auto")))

Edit any cell. Changes update EDITED_ROWS immediately.
🟢 green = verified  🟡 amber = fuzzy/medium  🔴 red = not found



## 💾 Cell 6 — Save CSV + JSON
Writes `<stem>_visits.csv` with every field **and** its `_quote`, `_unit`, `_ref_range`, `_confidence` columns. Also saves a full JSON for programmatic use.

In [6]:
import shutil

stem    = Path(REPORT_FILE).stem
out_dir = OUTPUT_DIR / stem
out_dir.mkdir(parents=True, exist_ok=True)

csv_path  = out_dir / f"{stem}_visits.csv"
json_path = out_dir / f"{stem}_visits.json"

# Backup
if csv_path.exists():
    bak = out_dir / f"{stem}_visits_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv.bak"
    shutil.copy2(csv_path, bak)
    print(f"💾 Backup → {bak.name}")

# ── CSV column order ──────────────────────────────────────────
# Meta | then for each field: value, unit, ref_range, quote, confidence
META_COLS = ["_note_index", "_note_date", "_dob", "_char_start", "_char_end"]
DATA_COLS = []
for f in LIVER_FIELDS + ["fib4_computed", "ast_alt_ratio"]:
    DATA_COLS += [f, f+"_unit", f+"_ref_range", f+"_quote", f+"_confidence"]

# fib4_computed and ast_alt_ratio have no unit/ref_range — that's fine, cells will be empty
ALL_COLS = META_COLS + DATA_COLS

with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=ALL_COLS, extrasaction="ignore")
    w.writeheader()
    for row in EDITED_ROWS:
        w.writerow({c: row.get(c, "") for c in ALL_COLS})

# ── Rename to target schema ───────────────────────────────────   ← INSERT HERE
    import pandas as pd
    rename_map = {
        "bilirubin_total" : "bilirubin",
        "fib4_computed"   : "fib4_score",
    }
    df = pd.read_csv(csv_path)
    df.rename(columns=rename_map, inplace=True)
    
    FINAL_COLS = [
        "subject_id", "gender", "age",
        "icd9_code", "short_title", "long_title", "liver_group",
        "ast", "alt", "ast_alt_ratio",
        "bilirubin", "albumin", "platelets", "inr",
        "creatinine", "glucose", "fib4_score",
    ]
    final_cols_present = [c for c in FINAL_COLS if c in df.columns]
    df_final = df[final_cols_present]
    
    final_path = out_dir / f"{stem}_final.csv"
    df_final.to_csv(final_path, index=False)
    print(f"  📄 Final schema CSV → {final_path}")

# ── JSON ──────────────────────────────────────────────────────   ← existing code continues
payload = {
    "generated_at" : datetime.now().isoformat(),
    ...
# ── JSON ──────────────────────────────────────────────────────
payload = {
    "generated_at" : datetime.now().isoformat(),
    "report_file"  : REPORT_FILE,
    "model"        : MODEL,
    "visit_count"  : len(EDITED_ROWS),
    "fields"       : LIVER_FIELDS,
    "visits"       : EDITED_ROWS,
}
json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Coverage stats ────────────────────────────────────────────
total  = len(EDITED_ROWS) * len(LIVER_FIELDS)
filled = sum(1 for r in EDITED_ROWS for f in LIVER_FIELDS
             if r.get(f) not in (None, "", "None"))
quoted = sum(1 for r in EDITED_ROWS for f in LIVER_FIELDS
             if r.get(f + "_quote") not in (None, "", "None"))
verified = sum(
    1 for r in EDITED_ROWS for f in LIVER_FIELDS
    if _sanity(r.get(f + "_quote"), FULL_TEXT) == "VERIFIED"
)

print(f"\n✅ Saved {len(EDITED_ROWS)} visit row(s)")
print(f"   Fields filled    : {filled:3d} / {total}  ({100*filled//max(total,1)}%)")
print(f"   Quotes present   : {quoted:3d} / {filled}  (hallucination traceability)")
print(f"   Quotes VERIFIED  : {verified:3d} / {quoted}")
print(f"\n  📄 CSV  → {csv_path}")
print(f"  📦 JSON → {json_path}")

SyntaxError: '{' was never closed (1496727530.py, line 56)

## 🗂️ Cell 6b — Per-Patient Split *(N patients)*
Groups `EDITED_ROWS` by **Date of Birth** (the `_dob` field set during segmentation).
Each unique DOB → its own sub-folder, CSV, and JSON.
If `_dob` is missing for a visit, it falls into an `unknown_dob` group.

**Outputs** (inside `OUTPUT_DIR / stem /`):
```
output/
  readable/
    readable_visits.csv          ← combined (Cell 6)
    readable_visits.json         ← combined (Cell 6)
    patient_1981-02-14/          ← one folder per DOB
      readable_visits_patient_1981-02-14.csv
      readable_visits_patient_1981-02-14.json
    patient_1969-03-20/
      readable_visits_patient_1969-03-20.csv
      readable_visits_patient_1969-03-20.json
```


In [ ]:
# ── Cell 6b — Per-Patient Split ───────────────────────────────────────────────
# Groups EDITED_ROWS by _dob. Each unique DOB gets its own CSV + JSON.
# Run after Cell 6 (relies on EDITED_ROWS, LIVER_FIELDS, stem, out_dir).

from collections import defaultdict

# ── Group rows by DOB ─────────────────────────────────────────
patient_groups = defaultdict(list)
for row in EDITED_ROWS:
    dob = (row.get("_dob") or "").strip() or "unknown_dob"
    patient_groups[dob].append(row)

n_patients = len(patient_groups)
print(f"🔍 Detected {n_patients} patient(s) by DOB:\n")
for dob, rows in sorted(patient_groups.items()):
    dates = [r.get('_note_date', '?') for r in rows]
    genders = list({r.get('gender', '?') for r in rows})
    ages    = list({r.get('age', '?') for r in rows})
    print(f"  DOB {dob:15s} → {len(rows)} visit(s)  "
          f"dates={dates}  gender={genders}  age={ages}")

# ── Column order (same as Cell 6) ─────────────────────────────
META_COLS = ["subject_id", "_note_index", "_note_date", "_dob", "_char_start", "_char_end"]

DATA_COLS = []
for f in LIVER_FIELDS + ["fib4_computed", "ast_alt_ratio"]:
    DATA_COLS += [f, f+"_unit", f+"_ref_range", f+"_quote", f+"_confidence"]

ALL_COLS = META_COLS + DATA_COLS

# ── Write one sub-folder per patient ──────────────────────────
print()
saved_files = []
for dob, rows in sorted(patient_groups.items()):
    safe_dob   = dob.replace("/", "-").replace(" ", "_")  # filesystem-safe
    pat_dir    = out_dir / f"patient_{safe_dob}"
    pat_dir.mkdir(parents=True, exist_ok=True)

    pat_csv  = pat_dir / f"{stem}_visits_patient_{safe_dob}.csv"
    pat_json = pat_dir / f"{stem}_visits_patient_{safe_dob}.json"

    # Backup if exists
    if pat_csv.exists():
        bak = pat_dir / f"{stem}_visits_patient_{safe_dob}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv.bak"
        shutil.copy2(pat_csv, bak)
        print(f"  💾 Backup → {bak.name}")

    # CSV
    with open(pat_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=ALL_COLS, extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow({c: row.get(c, "") for c in ALL_COLS})

    # JSON
    payload = {
        "generated_at" : datetime.now().isoformat(),
        "report_file"  : REPORT_FILE,
        "model"        : MODEL,
        "patient_dob"  : dob,
        "visit_count"  : len(rows),
        "fields"       : LIVER_FIELDS,
        "visits"       : rows,
    }
    pat_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    saved_files.append((dob, len(rows), pat_csv, pat_json))
    print(f"  ✅ DOB {dob:15s} — {len(rows)} visit(s)")
    print(f"      📄 {pat_csv}")
    print(f"      📦 {pat_json}")

print(f"\n🎉 Split complete — {n_patients} patient file(s) written.")


## 📈 Cell 7 — Disease Progression Plot *(optional)*
Plots key liver markers over time. Requires `matplotlib` and `python-dateutil`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dateutil.parser import parse as _parse_date

PLOT_FIELDS = [
    ("ast",           "AST (U/L)"),
    ("alt",           "ALT (U/L)"),
    ("fibroscan_kpa", "FibroScan (kPa)"),
    ("fib4_computed", "FIB-4 (computed)"),
    ("albumin",       "Albumin (g/dL)"),
    ("platelets",     "Platelets (k/uL)"),
    ("inr",           "INR"),
]

fig, axes = plt.subplots(len(PLOT_FIELDS), 1,
                         figsize=(11, 3.2 * len(PLOT_FIELDS)), sharex=True)
if len(PLOT_FIELDS) == 1:
    axes = [axes]

for ax, (field, ylabel) in zip(axes, PLOT_FIELDS):
    dates, vals = [], []
    for row in EDITED_ROWS:
        raw_d = row.get("_note_date") or row.get("visit_date")
        raw_v = row.get(field)
        if raw_d and raw_v not in (None, "", "None"):
            try:
                dates.append(_parse_date(str(raw_d)))
                vals.append(float(re.sub(r"[^\d.]", "", str(raw_v))))
            except Exception:
                pass
    if dates:
        ax.plot(dates, vals, marker="o", lw=2, ms=7, color="#2c7be5")
        for d, v in zip(dates, vals):
            ax.annotate(f"{v:.1f}", (d, v), textcoords="offset points",
                        xytext=(4, 4), fontsize=7.5)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.set_ylabel(ylabel, fontsize=9)
        ax.text(0.5, 0.5, f"no data for {field}", ha="center", va="center",
                transform=ax.transAxes, color="grey")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.suptitle(f"MASH Disease Progression — {Path(REPORT_FILE).stem}",
             y=1.005, fontsize=12, fontweight="bold")
plt.tight_layout()

plot_path = out_dir / f"{stem}_progression.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {plot_path}")